# Day 3 Final — Baseline ML: Vietnamese Price Prediction

**Dataset:** `SeanSunny/items_tv_v6` — lọc giá <= 1,000,000 VND  
**Split:** 85,727 train | 3,926 val | 3,872 test  
**Metric chính:** RMSLE (Root Mean Squared Logarithmic Error) — chuẩn Kaggle cho price prediction  
**Best kết quả:** Blended RMSLE = 0.5164

---

## Cấu trúc notebook

| Tier | Model | Mục đích |
|------|-------|----------|
| **1** | Random / Mean / Median | Baseline thống kê — không cần dữ liệu text |
| **2** | LR + Arch A / LR + Arch B | Baseline NLP — thấy vấn đề khi KHÔNG dùng log-transform |
| **3** | Ridge / LightGBM / LGB Tuned | ML chính — log1p + category feature |
| **4** | Weighted Blend | Kết hợp top models |

**Key techniques:**
- **Log-transform target:** train trên `log1p(price)`, predict `expm1(pred)` — tác động lớn nhất (-8.8% RMSLE)
- **Underthesea word segmentation:** tách từ tiếng Việt `"điện_thoại"`, `"màn_hình"` trước TF-IDF
- **Category feature:** one-hot 8 categories + TF-IDF (scipy sparse hstack)
- **Arch C:** kết hợp word bigram + char_wb 3-5gram — bắt brand names, model numbers, specs

In [ ]:
import sys
from pathlib import Path

# pricer_vi nằm ở thư mục cha (Data_processing_for_Vietnamese_data/)
sys.path.insert(0, str(Path().resolve().parent))

import random
import time
import pickle

import numpy as np
import pandas as pd
import plotly.express as px
from scipy.sparse import hstack
from scipy.optimize import minimize as scipy_minimize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb
from tqdm.auto import tqdm

from pricer_vi.items import Item
from pricer_vi.evaluator import rmsle, plot_predictions

SEED = 42
DATASET = "SeanSunny/items_tv_v6"
PRICE_THRESHOLD = 1_000_000
CACHE_DIR = Path().resolve()  # day3/
PLOT_SIZE = 200               # số điểm hiển thị trên chart (metric tính trên full test set)

random.seed(SEED)
np.random.seed(SEED)

print("Setup done.")

## 1. Load Data

Load từ HuggingFace Hub lần đầu, cache thành `.pkl` để lần sau chạy nhanh hơn.  
Filter giá `<= 1,000,000 VND` — loại bỏ 22.1% outlier đắt tiền, giảm skewness từ 6.85 → 1.23.

In [ ]:
items_train_cache = CACHE_DIR / "items_train.pkl"
items_val_cache   = CACHE_DIR / "items_val.pkl"
items_test_cache  = CACHE_DIR / "items_test.pkl"

if items_train_cache.exists():
    print("Loading items from local cache...")
    with open(items_train_cache, "rb") as f: train_raw = pickle.load(f)
    with open(items_val_cache,   "rb") as f: val_raw   = pickle.load(f)
    with open(items_test_cache,  "rb") as f: test_raw  = pickle.load(f)
else:
    print(f"Downloading from HuggingFace: {DATASET}")
    train_raw, val_raw, test_raw = Item.from_hub(DATASET)
    with open(items_train_cache, "wb") as f: pickle.dump(train_raw, f)
    with open(items_val_cache,   "wb") as f: pickle.dump(val_raw, f)
    with open(items_test_cache,  "wb") as f: pickle.dump(test_raw, f)
    print("Cached items to disk.")

print(f"Raw: {len(train_raw):,} train | {len(val_raw):,} val | {len(test_raw):,} test")

# Filter <= 1M VND
train = [item for item in train_raw if item.price <= PRICE_THRESHOLD]
val   = [item for item in val_raw   if item.price <= PRICE_THRESHOLD]
test  = [item for item in test_raw  if item.price <= PRICE_THRESHOLD]

print(f"Filtered <= {PRICE_THRESHOLD:,} VND:")
print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

# Numpy arrays hay dùng
train_prices = np.array([item.price for item in train], dtype=float)
val_prices   = np.array([item.price for item in val],   dtype=float)
test_prices  = np.array([item.price for item in test],  dtype=float)
test_names   = [item.title for item in test]

print(f"  Price range: {train_prices.min():,.0f} - {train_prices.max():,.0f} VND")
print(f"  Mean: {train_prices.mean():,.0f} | Median: {np.median(train_prices):,.0f}")

## 2. EDA — Exploratory Data Analysis

In [ ]:
# --- Price distribution ---
df_price = pd.DataFrame({"price": train_prices})
fig = px.histogram(
    df_price, x="price", nbins=60,
    title="Phân phối giá sản phẩm (Train, <= 1M VND)",
    labels={"price": "Giá (VND)"},
    width=800, height=400,
)
fig.update_xaxes(tickformat=",.0f")
fig.show()

# --- Category distribution ---
cat_counts = pd.Series([item.category for item in train]).value_counts().reset_index()
cat_counts.columns = ["category", "count"]
fig2 = px.bar(
    cat_counts, x="count", y="category", orientation="h",
    title="Số lượng sản phẩm theo danh mục (Train)",
    labels={"count": "Số sản phẩm", "category": ""},
    width=800, height=400,
)
fig2.show()

# --- Sample items ---
print("\nSample items:")
for item in train[:3]:
    print(f"  [{item.category}] {item.title[:60]} | {item.price:,} VND")
    print(f"    Summary: {item.summary[:100]}...")

## 3. Text Tokenization — Underthesea

Tiếng Việt cần tách từ trước khi đưa vào TF-IDF. `underthesea.word_tokenize()` nhận diện từ ghép:

| Raw | Sau tokenize |
|-----|-------------|
| `điện thoại thông minh` | `điện_thoại thông_minh` |
| `máy tính xách tay` | `máy_tính xách_tay` |
| `128gb pin 5000mah` | `128gb pin 5000mah` |

**Quan trọng:** underthesea KHÔNG thread-safe → phải pre-tokenize 1 lần và cache `.pkl`.  
Cache đã có sẵn từ run trước — load trực tiếp.

In [ ]:
from underthesea import word_tokenize
from multiprocessing import Pool

def tokenize_one(text):
    return word_tokenize(text, format="text")

def load_or_tokenize(cache_path, texts, desc):
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            result = pickle.load(f)
        print(f"Loaded cache: {cache_path.name} ({len(result):,} docs)")
        return result
    print(f"Tokenizing {len(texts):,} {desc} docs...")
    with Pool(4) as p:
        result = list(tqdm(p.imap(tokenize_one, texts, chunksize=500), total=len(texts)))
    with open(cache_path, "wb") as f:
        pickle.dump(result, f)
    print(f"Saved: {cache_path.name}")
    return result

tokenized_train = load_or_tokenize(
    CACHE_DIR / "tokenized_train_1m.pkl",
    [item.summary for item in train], "train"
)
tokenized_val = load_or_tokenize(
    CACHE_DIR / "tokenized_val_1m.pkl",
    [item.summary for item in val], "val"
)
tokenized_test = load_or_tokenize(
    CACHE_DIR / "tokenized_test_1m.pkl",
    [item.summary for item in test], "test"
)

print(f"\nSample raw:       {train[0].summary[:80]}")
print(f"Sample tokenized: {tokenized_train[0][:80]}")

## 4. Feature Engineering

### 4a. Text Vectorization — 3 Architectures

| Arch | Mô tả | max_features | Đặc điểm |
|------|-------|-------------|----------|
| **A** | TF-IDF raw text, ngram=(1,2), không tách từ | 10,000 | Đơn giản nhất |
| **B** | Underthesea word_tokenize → TF-IDF word unigram | 10,000 | Hiểu từ ghép tiếng Việt |
| **C** | Underthesea → FeatureUnion(word bigram 5K + char_wb 3-5gram 5K) | 10,000 | Bắt specs, brand, model number |

**char_wb** (character whitespace-bounded): bắt patterns như `"128gb"`, `"5000mah"`, `"samsung"` dù text bị viết tắt.

### 4b. Category Feature

8 danh mục → **OneHotEncoder** → 8 sparse features → `scipy.sparse.hstack` với TF-IDF matrix.

**Arch B + Cat** = `TF-IDF_B (10,000) + OneHot_Category (8)` = **10,008 features**  
**Arch C + Cat** = `TF-IDF_C (10,000) + OneHot_Category (8)` = **10,008 features**

### 4c. Log-transform Target

```
y_train = log1p(price)  →  train model  →  predict  →  expm1(pred) = price_vnd
```

RMSLE = sqrt(mean((log1p(pred) - log1p(true))²)) — metric này **đã ở log-space**.  
Train trực tiếp trên `log1p(price)` giúp model tối ưu đúng metric, tránh predict âm.

In [ ]:
# === Architecture A: TF-IDF raw text (bigram, no word segmentation) ===
t0 = time.time()
vectorizer_a = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))
train_docs_raw = [item.summary for item in train]
test_docs_raw  = [item.summary for item in test]
X_a_train = vectorizer_a.fit_transform(train_docs_raw)
X_a_test  = vectorizer_a.transform(test_docs_raw)
print(f"Arch A: {X_a_train.shape} ({time.time()-t0:.1f}s)")

# === Architecture B: Underthesea + TF-IDF word unigram ===
t0 = time.time()
vectorizer_b = TfidfVectorizer(max_features=10_000)
X_b_train = vectorizer_b.fit_transform(tokenized_train)
X_b_test  = vectorizer_b.transform(tokenized_test)
X_b_val   = vectorizer_b.transform(tokenized_val)
print(f"Arch B: {X_b_train.shape} ({time.time()-t0:.1f}s)")

# === Architecture C: word bigram + char_wb 3-5gram (FeatureUnion) ===
t0 = time.time()
arch_c = FeatureUnion([
    ("word", TfidfVectorizer(analyzer="word",    ngram_range=(1, 2), max_features=5_000)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=5_000)),
])
X_c_train = arch_c.fit_transform(tokenized_train)
X_c_test  = arch_c.transform(tokenized_test)
X_c_val   = arch_c.transform(tokenized_val)
print(f"Arch C: {X_c_train.shape} ({time.time()-t0:.1f}s)")

# === Category one-hot ===
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_cat_train = cat_encoder.fit_transform([[item.category] for item in train])
X_cat_val   = cat_encoder.transform([[item.category] for item in val])
X_cat_test  = cat_encoder.transform([[item.category] for item in test])
categories  = list(cat_encoder.categories_[0])
print(f"Category: {X_cat_train.shape[1]} features — {categories}")

# === Combined matrices (text + category) ===
X_bc_train = hstack([X_b_train, X_cat_train])
X_bc_test  = hstack([X_b_test,  X_cat_test])
X_bc_val   = hstack([X_b_val,   X_cat_val])

X_cc_train = hstack([X_c_train, X_cat_train])
X_cc_test  = hstack([X_c_test,  X_cat_test])
X_cc_val   = hstack([X_c_val,   X_cat_val])

print(f"Arch B+Cat: {X_bc_train.shape}")
print(f"Arch C+Cat: {X_cc_train.shape}")

# === Log-transform target ===
log_train_prices = np.log1p(train_prices)
log_val_prices   = np.log1p(val_prices)
print(f"\nLog price: {log_train_prices.min():.2f} - {log_train_prices.max():.2f}")
print(f"  Mean: {log_train_prices.mean():.2f} | Median: {np.median(log_train_prices):.2f}")

## 5. Tier 1 — Statistical Baselines

Không dùng text, không dùng ML — chỉ dựa vào thống kê phân phối giá train set.

| Model | Dự đoán | Ý nghĩa |
|-------|---------|----------|
| **Random** | Số ngẫu nhiên trong [min, max] | Lower bound tệ nhất |
| **Mean** | Mean(train_prices) = 301K VND | Trung bình đơn giản |
| **Median** | Median(train_prices) = 229K VND | Robust với outlier, baseline tốt nhất trong tier này |

> Chart: 200 điểm mẫu để quan sát. Metrics (RMSLE, MAE, MAPE, R2) tính trên toàn bộ 3,872 test items.

In [ ]:
results = {}  # thu thập kết quả tất cả model

# --- Random ---
min_price, max_price = int(train_prices.min()), int(train_prices.max())
rng = np.random.default_rng(SEED)
pred_random = rng.integers(min_price, max_price + 1, size=len(test)).astype(float)
results["Random"] = plot_predictions(
    test_prices, pred_random,
    title="Random",
    names=test_names, plot_size=PLOT_SIZE,
)

# --- Mean ---
pred_mean = np.full(len(test), train_prices.mean())
results["Mean"] = plot_predictions(
    test_prices, pred_mean,
    title=f"Mean (= {train_prices.mean():,.0f} VND)",
    names=test_names, plot_size=PLOT_SIZE,
)

# --- Median ---
pred_median = np.full(len(test), np.median(train_prices))
results["Median"] = plot_predictions(
    test_prices, pred_median,
    title=f"Median (= {np.median(train_prices):,.0f} VND)",
    names=test_names, plot_size=PLOT_SIZE,
)

## 6. Tier 2 — NLP Baseline: LR + TF-IDF (không log-transform)

Linear Regression train trực tiếp trên giá VND (raw). Đây là cách **sai** — mục đích là minh họa vấn đề.

### Tại sao LR không log-transform bị kém?

1. **Predict âm:** LR không bị ràng buộc output dương. Sản phẩm giá rẻ (< 50K VND) thường bị predict < 0.
2. **RMSLE scale:** RMSLE = sqrt(mean((log1p(pred) - log1p(true))²)) — penalize đều trên log scale.  
   Sai 100K trên sản phẩm 200K (50%) = cùng mức penalty với sai 500K trên sản phẩm 1M (50%).  
   Nhưng LR tối ưu MSE trên raw VND → không align với RMSLE.
3. **Skewed distribution:** Phân phối giá skewed (mean 301K >> median 229K) → MSE bị kéo về phía đắt.

**Arch A** (raw bigram, không tách từ tiếng Việt) vs **Arch B** (Underthesea word segmentation):  
Arch B tốt hơn vì từ ghép như `"điện_thoại"`, `"thông_minh"` được nhận diện đúng.

In [ ]:
# === LR + Arch A (raw text bigram, NO log-transform) ===
t0 = time.time()
lr_a = LinearRegression()
lr_a.fit(X_a_train, train_prices)  # train trên raw VND
print(f"LR Arch A train: {time.time()-t0:.1f}s")

pred_lr_a = np.clip(lr_a.predict(X_a_test), 0, None)  # clip âm về 0
results["LR + Arch A"] = plot_predictions(
    test_prices, pred_lr_a,
    title="LR + Arch A (TF-IDF bigram, raw, no word segmentation)",
    names=test_names, plot_size=PLOT_SIZE,
)

# === LR + Arch B (Underthesea word tokenize, NO log-transform) ===
t0 = time.time()
lr_b = LinearRegression()
lr_b.fit(X_b_train, train_prices)  # train trên raw VND
print(f"LR Arch B train: {time.time()-t0:.1f}s")

pred_lr_b = np.clip(lr_b.predict(X_b_test), 0, None)
results["LR + Arch B"] = plot_predictions(
    test_prices, pred_lr_b,
    title="LR + Arch B (TF-IDF + Underthesea word segmentation, raw)",
    names=test_names, plot_size=PLOT_SIZE,
)

## 7. Tier 3 — ML Models: Log-transform + Category Feature

### 7a. Ridge Regression

**Ridge** = Linear Regression + L2 regularization: `loss = MSE + alpha * ||w||²`

- `alpha=1.0`: penalize weight lớn → tránh overfitting trên 10K sparse features
- Bất ngờ cạnh tranh với GBDT (chỉ kém LGB 2.4%) vì TF-IDF sparse matrix phù hợp với linear model
- Train **trên `log1p(price)`** → predict **`expm1(pred)`** → RMSLE cải thiện rõ rệt

In [ ]:
# === Ridge + Arch B + Cat (log-transform) ===
# Arch B + Cat = TF-IDF Underthesea (10K) + OneHot Category (8) = 10,008 features
t0 = time.time()
ridge = Ridge(alpha=1.0)
ridge.fit(X_bc_train, log_train_prices)  # train trên log1p(price)
print(f"Ridge (B+Cat, log) train: {time.time()-t0:.1f}s")

pred_ridge = np.clip(np.expm1(ridge.predict(X_bc_test)), 0, None)
results["Ridge + Arch B+Cat"] = plot_predictions(
    test_prices, pred_ridge,
    title="Ridge + Arch B+Cat (alpha=1.0, log1p target)",
    names=test_names, plot_size=PLOT_SIZE,
)

### 7b. LightGBM Default

**LightGBM** (Gradient Boosting Decision Tree) — nhanh nhất trong họ GBDT nhờ:
- **Leaf-wise tree growth** (thay vì level-wise của XGBoost): tìm leaf có gain lớn nhất để split
- **Histogram-based splitting**: bin continuous features → tìm split điểm trong O(bins) thay O(N)

Config default:
- `n_estimators=1000`: 1000 cây
- `learning_rate=0.1`: bước học — shrinkage để tránh overfit
- `n_jobs=-1`: dùng tất cả CPU cores

In [ ]:
# === LightGBM + Arch B + Cat (log-transform, default params) ===
t0 = time.time()
lgb_default = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
lgb_default.fit(X_bc_train, log_train_prices)
print(f"LightGBM (B+Cat, log) train: {time.time()-t0:.1f}s")

pred_lgb_default = np.clip(np.expm1(lgb_default.predict(X_bc_test)), 0, None)
results["LightGBM + Arch B+Cat"] = plot_predictions(
    test_prices, pred_lgb_default,
    title="LightGBM + Arch B+Cat (default, log1p target)",
    names=test_names, plot_size=PLOT_SIZE,
)

### 7c. LightGBM Tuned (Arch C + Cat)

Best params từ Optuna 50 trials (val RMSLE = 0.5588 sau 1h28m tuning — hardcode để tái sử dụng):

| Param | Giá trị | Ý nghĩa |
|-------|---------|----------|
| `num_leaves` | 173 | Độ phức tạp cây — càng cao càng fit train tốt hơn |
| `min_child_samples` | 50 | Số mẫu tối thiểu mỗi leaf — regularize |
| `feature_fraction` | 0.689 | Dùng 68.9% features mỗi cây — giảm correlation giữa cây |
| `lambda_l1` | 0.011 | L1 regularization — sparsity |
| `lambda_l2` | 0.155 | L2 regularization — smooth weights |
| `learning_rate` | 0.032 | Nhỏ hơn default (0.1) → cần nhiều cây hơn nhưng generalize tốt hơn |
| `n_estimators` | 1500 | Tăng từ 1000 vì lr nhỏ hơn |

**Arch C + Cat** = `FeatureUnion(word bigram 5K + char_wb 3-5gram 5K)` + OneHot Category (8) = **10,008 features**  
Arch C tốt hơn Arch B sau tuning vì char_wb bắt được `"samsung"`, `"128gb"`, `"iphone_14"` dù viết không dấu.

In [ ]:
# === LightGBM Tuned + Arch C + Cat ===
# Best params từ Optuna 50 trials (day3 v3, val RMSLE=0.5588)
BEST_LGB_PARAMS = {
    "num_leaves":        173,
    "min_child_samples": 50,
    "feature_fraction":  0.689,
    "lambda_l1":         0.011,
    "lambda_l2":         0.155,
    "learning_rate":     0.032,
    "n_estimators":      1500,
    "random_state":      SEED,
    "n_jobs":            -1,
    "verbose":           -1,
}

t0 = time.time()
lgb_tuned = lgb.LGBMRegressor(**BEST_LGB_PARAMS)
lgb_tuned.fit(X_cc_train, log_train_prices)
print(f"LightGBM Tuned (C+Cat, log) train: {time.time()-t0:.1f}s")

pred_lgb_tuned = np.clip(np.expm1(lgb_tuned.predict(X_cc_test)), 0, None)
results["LightGBM Tuned + Arch C+Cat"] = plot_predictions(
    test_prices, pred_lgb_tuned,
    title="LightGBM Tuned + Arch C+Cat (Optuna best params, log1p target)",
    names=test_names, plot_size=PLOT_SIZE,
)

## 8. Tier 4 — Weighted Blending

Kết hợp predictions từ nhiều model để giảm variance. Dùng `scipy.optimize.minimize` (Nelder-Mead)  
tìm weights tối ưu trên **validation set**, sau đó áp dụng lên test set.

**Tại sao blending cải thiện ít (chỉ -1%)?**  
Các model này có correlation cao (cùng dùng TF-IDF sparse, cùng data) → diversity thấp.  
Blending hiệu quả nhất khi models có error pattern khác nhau (Day 4 sẽ dùng dense embeddings để tăng diversity).

Blend 3 models tốt nhất: LightGBM Tuned C+Cat, LightGBM B+Cat, Ridge B+Cat.

In [ ]:
# Predictions trên val set để tìm weights tối ưu
pred_lgb_tuned_val  = np.clip(np.expm1(lgb_tuned.predict(X_cc_val)),   0, None)
pred_lgb_default_val = np.clip(np.expm1(lgb_default.predict(X_bc_val)), 0, None)
pred_ridge_val      = np.clip(np.expm1(ridge.predict(X_bc_val)),        0, None)

val_preds_list = [pred_lgb_tuned_val, pred_lgb_default_val, pred_ridge_val]
blend_names    = ["LightGBM Tuned C+Cat", "LightGBM B+Cat", "Ridge B+Cat"]

def blend_rmsle(weights):
    w = np.abs(weights)
    w = w / w.sum()
    blended = sum(wi * p for wi, p in zip(w, val_preds_list))
    return rmsle(val_prices, blended)

result_opt = scipy_minimize(blend_rmsle, x0=[0.5, 0.3, 0.2], method="Nelder-Mead")
best_weights = np.abs(result_opt.x)
best_weights /= best_weights.sum()

print("Optimal weights (minimized on val set):")
for name, w in zip(blend_names, best_weights):
    print(f"  {name}: {w:.3f}")
print(f"Val blend RMSLE: {result_opt.fun:.4f}")

# Áp dụng weights lên test set
test_preds_list = [pred_lgb_tuned, pred_lgb_default, pred_ridge]
pred_blended = sum(w * p for w, p in zip(best_weights, test_preds_list))

results["Blended (3 models)"] = plot_predictions(
    test_prices, pred_blended,
    title="Blended: LGB Tuned C + LGB B + Ridge (weights from val)",
    names=test_names, plot_size=PLOT_SIZE,
)

## 9. Final Results

Toàn bộ metrics tính trên **3,872 test items** (full test set, không phải 200).  
Bảng sắp xếp theo RMSLE tăng dần.

In [ ]:
print("=" * 80)
print(f"FINAL RESULTS — Day 3 Final (Test: {len(test):,} items, <= {PRICE_THRESHOLD:,} VND)")
print("=" * 80)
print(f"\n{'Model':<35} {'RMSLE':>8} {'MAE (VND)':>13} {'MAPE':>8} {'R2':>7}")
print("-" * 78)

sorted_results = sorted(results.items(), key=lambda x: x[1]["rmsle"])
for name, res in sorted_results:
    print(
        f"{name:<35} {res['rmsle']:>8.4f} {res['mae']:>13,.0f} "
        f"{res['mape']:>7.1f}% {res['r2']:>6.1f}%"
    )

best_name, best_res = sorted_results[0]
print(f"\nBest model: {best_name} — RMSLE={best_res['rmsle']:.4f}")

# So sánh log-transform impact
lr_b_rmsle     = results["LR + Arch B"]["rmsle"]
lgb_best_rmsle = best_res["rmsle"]
print(f"\nLog-transform + GBDT vs LR raw:")
print(f"  LR + Arch B (no log): {lr_b_rmsle:.4f}")
print(f"  {best_name}: {lgb_best_rmsle:.4f}")
print(f"  Improvement: {(lr_b_rmsle - lgb_best_rmsle) / lr_b_rmsle * 100:.1f}%")

print("\n" + "=" * 80)
print("Day 3 ceiling: RMSLE ~ 0.52 (TF-IDF bag-of-words limit)")
print("Day 4: Dense embeddings (PhoBERT, AITeamVN) -> target RMSLE <= 0.40")
print("=" * 80)

## 10. Phân tích & Bài học

### Impact của từng cải tiến

| Cải tiến | RMSLE trước → sau | % giảm | Impact |
|----------|-------------------|--------|--------|
| **Log-transform** | ~0.57 → ~0.52 | -8.8% | **Lớn nhất** |
| **Category feature** | ~0.52 → ~0.50 | ~4% | Trung bình |
| **Arch C (char_wb)** | 0.5286 → 0.5217 | -1.3% | Nhỏ |
| **Blending** | 0.5217 → 0.5164 | -1.0% | Nhỏ |

### Bài học chính

1. **Log-transform bắt buộc** khi dùng RMSLE — train thẳng trên log-space giúp align model với metric.
2. **Ridge cạnh tranh với LGB** trên TF-IDF sparse (chỉ kém 2.4%) — L2 regularization quan trọng với 10K features.
3. **Arch C cần tuning** — char_wb thêm noise khi default params, nhưng Optuna khai thác được.
4. **Blending diminishing returns** khi models có correlation cao (cùng pipeline).
5. **Ceiling TF-IDF ~0.52** — TF-IDF mất thứ tự từ, ngữ nghĩa, ngữ cảnh → cần dense embeddings.

### Hướng Day 4

| Approach | Model | Expected RMSLE |
|----------|-------|----------------|
| Frozen embeddings | AITeamVN (BGE-M3 1024d) + MLP | ~0.50 |
| Fine-tune | PhoBERT-base-v2 full fine-tune | ~0.44 |
| Fine-tune + LLRD + EMA | PhoBERT++ | ~0.43 |
| Stacking 7 models | Ridge + ElasticNet + LGB meta | **~0.40** |